In [1]:
import os
import glob
import random
import warnings
import multiprocessing
import pandas as pd
import numpy as np
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio.transforms as T
import pytorch_lightning as pl
from torch.utils.data import Dataset, DataLoader
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_lightning.loggers import WandbLogger
from torchmetrics.classification import MulticlassF1Score
from transformers import ASTConfig, ASTModel, get_cosine_schedule_with_warmup
from pathlib import Path
from tqdm import tqdm
import concurrent.futures

In [2]:
!uv pip install --upgrade wandb

Using Python 3.12.12 environment at: /usr
Resolved 20 packages in 383ms                                        
Prepared 10 packages in 497ms                                            
Uninstalled 10 packages in 91ms
Installed 10 packages in 34ms                               
 - certifi==2026.1.4
 + certifi==2026.2.25
 - packaging==25.0
 + packaging==26.0
 - platformdirs==4.5.1
 + platformdirs==4.9.2
 - protobuf==5.29.5
 + protobuf==6.33.5
 - pydantic==2.12.3
 + pydantic==2.12.5
 - pydantic-core==2.41.4
 + pydantic-core==2.41.5
 - requests==2.32.4
 + requests==2.32.5
 - sentry-sdk==2.49.0
 + sentry-sdk==2.54.0
 - urllib3==2.5.0
 + urllib3==2.6.3
 - wandb==0.24.0
 + wandb==0.25.0


In [3]:
# ==========================================
# 0. CONFIGURATION & SETUP
# ==========================================
warnings.filterwarnings('ignore')
pl.seed_everything(14, workers=True)

# Toggle pipeline stages here
RUN_GENERATION = False
RUN_TRAINING = False
RUN_INFERENCE = True

CONFIG = {
    "project_name": "23f2001635-t12026",
    "experiment_name": "ast_mashup_v1",
    "data_root": "/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup",
    "save_dir": "/kaggle/input/datasets/harshitsahu01/dataset-for-ast-v1/processed_data_ast",
    "best_checkpoint":"/kaggle/input/models/harshitsahu01/ast-genre-predictor/pytorch/default/1/checkpoints/best-ast-model.ckpt",
    "lr": 5e-5,              # AST requires low LR
    "weight_decay": 0.05,    # High weight decay for transformers
    "batch_size": 16,         # AST is large; keep batch size small (use accumulate_grad)
    "accumulate_grad": 4,    # Effective batch size = 64
    "num_epochs": 50,
    "patience": 5,           # Early stopping patience
    "num_classes": 10,
    "sample_rate": 16000,
    "duration": 30,          # Generate full 30s
    "ast_frames": 1024,      # 10.24 seconds required by AST
    "train_samples": 10000,
    "val_samples": 1500,
    "num_workers": multiprocessing.cpu_count(),
}

GENRES = ["blues", "classical", "country", "disco", "hiphop", 
          "jazz", "metal", "pop", "reggae", "rock"]
GENRE_TO_IDX = {g: i for i, g in enumerate(GENRES)}
IDX_TO_GENRE = {i: g for i, g in enumerate(GENRES)}

# AST Specific Normalization (AudioSet Mean/Std)
AST_MEAN = -4.2677393
AST_STD = 4.5689974

if RUN_TRAINING:
    import wandb
    from kaggle_secrets import UserSecretsClient
    try:
        user_secrets = UserSecretsClient()
        wandb.login(key=user_secrets.get_secret("WANDB_API_KEY"))
    except Exception as e:
        print(f"WandB login skipped/failed: {e}")

Seed set to 14


In [4]:
# ==========================================
# 1. DATA GENERATION (CPU MULTIPROCESSING)
# ==========================================

def get_random_stem(data_root, genre, stem_name, sample_rate):
    genre_dir = Path(data_root) / "genres_stems" / genre
    song_folders = [f.path for f in os.scandir(genre_dir) if f.is_dir()]
    stem_path = Path(random.choice(song_folders)) / f"{stem_name}.wav"
    audio, _ = librosa.load(stem_path, sr=sample_rate)
    return audio

def get_random_noise(data_root, sample_rate):
    noise_dir = Path(data_root) / "ESC-50-master" / "audio"
    noise_files = glob.glob(str(noise_dir / "*.wav"))
    noise_audio, _ = librosa.load(random.choice(noise_files), sr=sample_rate)
    return noise_audio

def process_single_mashup(sample_id, genre, config):
    try:
        data_root, sr = config["data_root"], config["sample_rate"]
        target_samples = config["duration"] * sr
        
        # 1. Load Stems & Tempo Match (Simplified for speed/robustness)
        stems = []
        for stem_type in ["drums", "bass", "vocals", "other"]:
            audio = get_random_stem(data_root, genre, stem_type, sr)
            if len(audio) > target_samples:
                start = random.randint(0, len(audio) - target_samples)
                audio = audio[start:start+target_samples]
            else:
                audio = np.pad(audio, (0, target_samples - len(audio)))
            stems.append(audio * random.uniform(0.6, 1.4))
            
        mixed_audio = np.sum(stems, axis=0)

        # 2. Inject Noise
        noise = get_random_noise(data_root, sr)
        if len(noise) < len(mixed_audio):
            noise = np.tile(noise, int(np.ceil(len(mixed_audio) / len(noise))))
        noise = noise[:len(mixed_audio)]
        
        target_snr = random.uniform(5, 20)
        rms_sig, rms_noise = np.sqrt(np.mean(mixed_audio**2)+1e-6), np.sqrt(np.mean(noise**2)+1e-6)
        mixed_audio += noise * (rms_sig / rms_noise) * (10 ** (-target_snr / 20))

        # 3. Create AST-Compliant Spectrogram (25ms window, 10ms hop)
        mel_spec = librosa.feature.melspectrogram(
            y=mixed_audio, sr=sr, n_mels=128, n_fft=400, hop_length=160
        )
        mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
        
        # Normalize with AST AudioSet stats
        mel_spec_db = (mel_spec_db - AST_MEAN) / (AST_STD * 2)
        
        tensor = torch.tensor(mel_spec_db, dtype=torch.float32) # Shape: (128, ~3000)
        
        prefix = "train" if "train" in sample_id else "val"
        save_dir = Path(config["save_dir"]) / prefix / genre
        save_dir.mkdir(parents=True, exist_ok=True)
        torch.save(tensor, save_dir / f"{sample_id}.pt")
        return True

    except Exception as e:
        return False

def generate_data():
    print("--- GENERATING SYNTHETIC MASHUPS (AST FORMAT) ---")
    tasks = []
    buffer = 50 
    for genre in GENRES:
        for i in range(CONFIG["train_samples"] // CONFIG["num_classes"] + buffer):
            tasks.append((f"train_{genre}_{i}", genre, CONFIG))
        for i in range(CONFIG["val_samples"] // CONFIG["num_classes"] + buffer):
            tasks.append((f"val_{genre}_{i}", genre, CONFIG))

    success = 0
    with concurrent.futures.ProcessPoolExecutor(max_workers=CONFIG['num_workers']) as executor:
        futures = [executor.submit(process_single_mashup, *task) for task in tasks]
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(tasks)):
            if future.result(): success += 1
    print(f"Generated {success} tensors.")

In [5]:
# ==========================================
# 2. DATASET & DYNAMIC AUGMENTATION
# ==========================================

class ASTMashupDataset(Dataset):
    def __init__(self, data_dir, config, is_train=True):
        self.files = glob.glob(f"{data_dir}/*/*.pt")
        self.is_train = is_train
        self.target_frames = config["ast_frames"]
        
        self.time_masking = T.TimeMasking(time_mask_param=100)
        self.freq_masking = T.FrequencyMasking(freq_mask_param=24)

    def __len__(self): return len(self.files)

    def __getitem__(self, idx):
        file_path = self.files[idx]
        label = GENRE_TO_IDX[Path(file_path).parent.name]
        mel_spec = torch.load(file_path) # Shape (128, Time)
        
        # Random Temporal Cropping (10.24 seconds)
        total_frames = mel_spec.shape[1]
        if total_frames > self.target_frames:
            if self.is_train:
                start = random.randint(0, total_frames - self.target_frames)
            else:
                start = (total_frames - self.target_frames) // 2 # Center crop for val
            mel_spec = mel_spec[:, start : start + self.target_frames]
        else:
            mel_spec = F.pad(mel_spec, (0, self.target_frames - total_frames))

        # AST expects input shape: (batch_size, 1024, 128)
        mel_spec = mel_spec.transpose(0, 1)

        if self.is_train:
            # Torchaudio expects (..., freq, time), so transpose temporarily
            mel_spec = mel_spec.transpose(0, 1).unsqueeze(0)
            mel_spec = self.time_masking(mel_spec)
            mel_spec = self.freq_masking(mel_spec)
            mel_spec = mel_spec.squeeze(0).transpose(0, 1)

        return mel_spec, torch.tensor(label, dtype=torch.long)

In [6]:
# ==========================================
# 3. AST MODEL & FOCAL LOSS
# ==========================================

class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        return (self.alpha * (1 - pt)**self.gamma * ce_loss).mean()

class MessyMashupAST(pl.LightningModule):
    def __init__(self, config):
        super().__init__()
        self.save_hyperparameters(config)
        self.config = config
        
        # Load Pre-trained AudioSet AST
        self.ast = ASTModel.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")
        
        # Custom Classification Head
        self.classifier = nn.Sequential(
            nn.LayerNorm(768),
            nn.Dropout(0.5),
            nn.Linear(768, config["num_classes"])
        )

        self.criterion = FocalLoss(gamma=2.0)
        self.train_f1 = MulticlassF1Score(num_classes=config["num_classes"], average='macro')
        self.val_f1 = MulticlassF1Score(num_classes=config["num_classes"], average='macro')

    def forward(self, x):
        # x shape: (batch, 1024, 128)
        outputs = self.ast(x)
        cls_token = outputs.last_hidden_state[:, 0, :] # Extract [CLS] token
        return self.classifier(cls_token)

    def training_step(self, batch, batch_idx):
        x, y = batch
        
        # --- MIXUP IMPLEMENTATION ---
        if np.random.rand() < 0.5:
            alpha = 0.4
            lam = np.random.beta(alpha, alpha)
            rand_index = torch.randperm(x.size()[0]).cuda()
            target_a, target_b = y, y[rand_index]
            x = lam * x + (1 - lam) * x[rand_index]
            
            logits = self(x)
            loss = lam * self.criterion(logits, target_a) + (1 - lam) * self.criterion(logits, target_b)
        else:
            logits = self(x)
            loss = self.criterion(logits, y)

        preds = torch.argmax(logits, dim=1)
        self.train_f1(preds, y)
        self.log('train_loss', loss, prog_bar=True)
        self.log('train_macro_f1', self.train_f1, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        preds = torch.argmax(logits, dim=1)
        self.val_f1(preds, y)
        self.log('val_loss', self.criterion(logits, y), prog_bar=True)
        self.log('val_macro_f1', self.val_f1, prog_bar=True)

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            self.parameters(), 
            lr=self.config["lr"], 
            weight_decay=self.config["weight_decay"]
        )
        # Linear Warmup + Cosine Decay
        total_steps = self.trainer.estimated_stepping_batches
        scheduler = get_cosine_schedule_with_warmup(
            optimizer, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps
        )
        return {
            "optimizer": optimizer, 
            "lr_scheduler": {"scheduler": scheduler, "interval": "step"}
        }

In [10]:
# ==========================================
# 4. INFERENCE ENGINE
# ==========================================

def run_inference(model, config):
    print("--- RUNNING INFERENCE ---")
    model.eval()
    model.cuda()
    
    test_csv_path = f"{config['data_root']}/test.csv"
    test_df = pd.read_csv(test_csv_path)
    predictions = []
    
    for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
        file_id = row['id']
        filename = row['filename']
        audio_path = f"{config['data_root']}/{filename}" 
        
        try:
            y, _ = librosa.load(audio_path, sr=config["sample_rate"])
            
            mel_spec = librosa.feature.melspectrogram(
                y=y, sr=config["sample_rate"], n_mels=128, n_fft=400, hop_length=160
            )
            mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
            mel_spec_db = (mel_spec_db - AST_MEAN) / (AST_STD * 2)
            
            tensor = torch.tensor(mel_spec_db, dtype=torch.float32)
            
            # Center crop 10.24 seconds for inference
            total_frames = tensor.shape[1]
            if total_frames > config["ast_frames"]:
                start = (total_frames - config["ast_frames"]) // 2
                tensor = tensor[:, start : start + config["ast_frames"]]
            else:
                tensor = F.pad(tensor, (0, config["ast_frames"] - total_frames))
                
            tensor = tensor.transpose(0, 1).unsqueeze(0).cuda() # (1, 1024, 128)

            with torch.no_grad():
                logits = model(tensor)
                pred_idx = torch.argmax(logits, dim=1).item()
                predictions.append(IDX_TO_GENRE[pred_idx])
                
        except Exception as e:
            print(f"Failed on {file_id}: {e}")
            predictions.append("pop") 

    test_df['genre'] = predictions
    test_df[['id', 'genre']].to_csv('submission.csv', index=False)
    print("Saved predictions to submission.csv")

In [11]:
# ==========================================
# 5. EXECUTION PIPELINE
# ==========================================

if __name__ == "__main__":
    if RUN_GENERATION:
        generate_data()

    if RUN_TRAINING:
        print("--- STARTING TRAINING ---")
        train_ds = ASTMashupDataset(f"{CONFIG['save_dir']}/train", CONFIG, is_train=True)
        val_ds = ASTMashupDataset(f"{CONFIG['save_dir']}/val", CONFIG, is_train=False)
        
        train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=4)
        val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=4)
        
        model = MessyMashupAST(CONFIG)
        
        checkpoint_cb = ModelCheckpoint(
            dirpath="checkpoints", filename="best-ast-model", monitor="val_macro_f1", mode="max", save_top_k=1
        )
        early_stop_cb = EarlyStopping(
            monitor="val_macro_f1", patience=CONFIG["patience"], mode="max", verbose=True
        )
        
        wandb_logger = WandbLogger(project=CONFIG["project_name"], name=CONFIG["experiment_name"])
        
        trainer = pl.Trainer(
            max_epochs=CONFIG["num_epochs"],
            accelerator="gpu",
            logger=wandb_logger,
            callbacks=[checkpoint_cb, early_stop_cb],
            accumulate_grad_batches=CONFIG["accumulate_grad"],
            precision="16-mixed",
            deterministic=True
        )
        
        trainer.fit(model, train_loader, val_loader)
        
    if RUN_INFERENCE:
        checkpoint_path = CONFIG['best_checkpoint']
        if os.path.exists(checkpoint_path):
            print(f"Loading best checkpoint: {checkpoint_path}")
            model = MessyMashupAST.load_from_checkpoint(checkpoint_path)
        else:
            print("No checkpoint found! Initializing untrained model.")
            model = MessyMashupAST(CONFIG)
            
        run_inference(model, CONFIG)

Loading best checkpoint: /kaggle/input/models/harshitsahu01/ast-genre-predictor/pytorch/default/1/checkpoints/best-ast-model.ckpt


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

ASTModel LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                         | Status     |  | 
----------------------------+------------+--+-
classifier.layernorm.bias   | UNEXPECTED |  | 
classifier.dense.weight     | UNEXPECTED |  | 
classifier.dense.bias       | UNEXPECTED |  | 
classifier.layernorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


--- RUNNING INFERENCE ---


  4%|▍         | 129/3020 [00:17<06:29,  7.42it/s]


KeyboardInterrupt: 